In [ ]:
import random


# ============================================================
# Genetic Algorithm for Exam Hall Seating
# ============================================================

class ExamSeatingGA:

    def __init__(self, students, rows, cols,
                 population_size=100,
                 generations=500,
                 crossover_rate=0.8,
                 mutation_rate=0.2):

        self.students = students
        self.rows = rows
        self.cols = cols

        self.population_size = population_size
        self.generations = generations
        self.crossover_rate = crossover_rate
        self.mutation_rate = mutation_rate

        self.n = len(students)

    # --------------------------------------------------------
    # Create random chromosome
    # --------------------------------------------------------
    def create_chromosome(self):
        chromosome = list(range(self.n))
        random.shuffle(chromosome)
        return chromosome

    # --------------------------------------------------------
    # Create initial population
    # --------------------------------------------------------
    def create_population(self):
        return [
            self.create_chromosome()
            for _ in range(self.population_size)
        ]

    # --------------------------------------------------------
    # Count adjacent same-subject pairs
    # --------------------------------------------------------
    def count_conflicts(self, chromosome):

        conflicts = 0

        for r in range(self.rows):
            for c in range(self.cols):

                index = r * self.cols + c

                # Current student's subject
                current_subject = self.students[chromosome[index]]

                # Check right neighbor
                if c < self.cols - 1:
                    right_index = r * self.cols + (c + 1)

                    right_subject = self.students[
                        chromosome[right_index]
                    ]

                    if current_subject == right_subject:
                        conflicts += 1

                # Check bottom neighbor
                if r < self.rows - 1:
                    bottom_index = (r + 1) * self.cols + c

                    bottom_subject = self.students[
                        chromosome[bottom_index]
                    ]

                    if current_subject == bottom_subject:
                        conflicts += 1

        return conflicts

    # --------------------------------------------------------
    # Fitness
    # --------------------------------------------------------
    def fitness(self, chromosome):

        conflicts = self.count_conflicts(chromosome)

        return 1 / (1 + conflicts)

    # --------------------------------------------------------
    # Tournament Selection
    # --------------------------------------------------------
    def selection(self, population):

        tournament_size = 3

        selected = random.sample(
            population,
            tournament_size
        )

        return max(
            selected,
            key=self.fitness
        )

    # --------------------------------------------------------
    # Order Crossover (OX)
    # --------------------------------------------------------
    def order_crossover(self, parent1, parent2):

        size = len(parent1)

        start, end = sorted(
            random.sample(range(size), 2)
        )

        child = [None] * size

        # Copy section from parent 1
        child[start:end + 1] = parent1[start:end + 1]

        # Fill remaining positions from parent 2
        remaining = [
            gene for gene in parent2
            if gene not in child
        ]

        position = 0

        for gene in remaining:

            while child[position] is not None:
                position += 1

            child[position] = gene

        return child

    # --------------------------------------------------------
    # Swap Mutation
    # --------------------------------------------------------
    def mutation(self, chromosome):

        if random.random() < self.mutation_rate:

            i, j = random.sample(
                range(self.n),
                2
            )

            chromosome[i], chromosome[j] = (
                chromosome[j],
                chromosome[i]
            )

        return chromosome

    # --------------------------------------------------------
    # Genetic Algorithm
    # --------------------------------------------------------
    def run(self):

        population = self.create_population()

        best_solution = None
        best_fitness = -1

        for generation in range(self.generations):

            # Find best chromosome
            current_best = max(
                population,
                key=self.fitness
            )

            current_fitness = self.fitness(
                current_best
            )

            # Update global best
            if current_fitness > best_fitness:

                best_solution = current_best[:]
                best_fitness = current_fitness

            # Stop if there are no conflicts
            if self.count_conflicts(best_solution) == 0:
                break

            # New population
            new_population = []

            # Elitism
            new_population.append(
                best_solution[:]
            )

            while len(new_population) < self.population_size:

                parent1 = self.selection(population)
                parent2 = self.selection(population)

                # Crossover
                if random.random() < self.crossover_rate:

                    child = self.order_crossover(
                        parent1,
                        parent2
                    )

                else:
                    child = parent1[:]

                # Mutation
                child = self.mutation(child)

                new_population.append(child)

            population = new_population

        return best_solution


# ============================================================
# Display Seating Chart
# ============================================================

def display_seating(chromosome, students, rows, cols):

    print("\n========== FINAL SEATING CHART ==========\n")

    for r in range(rows):

        for c in range(cols):

            index = r * cols + c
            student_number = chromosome[index]

            print(
                f"S{student_number + 1}"
                f"({students[student_number]})",
                end="\t"
            )

        print()


# ============================================================
# MAIN PROGRAM
# ============================================================

print("==========================================")
print("       EXAM HALL SEATING - GA")
print("==========================================")

# ------------------------------------------------------------
# User Input
# ------------------------------------------------------------

rows = int(input("Enter number of rows: "))
cols = int(input("Enter number of columns: "))

total_students = rows * cols

print("\nTotal seats:", total_students)

print("\nEnter subject code for each student.")
print("Example: CS, EC, ME, EE\n")

students = []

for i in range(total_students):

    subject = input(
        f"Enter subject code for Student {i + 1}: "
    ).strip().upper()

    students.append(subject)


# ------------------------------------------------------------
# GA Parameters
# ------------------------------------------------------------

population_size = int(
    input("\nEnter population size: ")
)

generations = int(
    input("Enter number of generations: ")
)

mutation_rate = float(
    input("Enter mutation rate (0-1): ")
)

crossover_rate = float(
    input("Enter crossover rate (0-1): ")
)


# ------------------------------------------------------------
# Create GA
# ------------------------------------------------------------

ga = ExamSeatingGA(
    students=students,
    rows=rows,
    cols=cols,
    population_size=population_size,
    generations=generations,
    mutation_rate=mutation_rate,
    crossover_rate=crossover_rate
)


# ------------------------------------------------------------
# Run GA
# ------------------------------------------------------------

best_solution = ga.run()


# ------------------------------------------------------------
# Output
# ------------------------------------------------------------

conflicts = ga.count_conflicts(best_solution)
fitness = ga.fitness(best_solution)

display_seating(
    best_solution,
    students,
    rows,
    cols
)

print("\n==========================================")
print("Number of conflicts:", conflicts)
print("Fitness:", fitness)
print("==========================================")


       EXAM HALL SEATING - GA
